# Chit 3 baseline: flagging RFI over the Karoo · **GPU-RT**
**AIMS Edge ML for Science · Capstone baseline** · 🟩 AI-on

Score every pixel of a HERA waterfall as RFI or not, first with the **classical threshold flagger** (the baseline to beat), then with a tiny U-Net. Measure per-pixel AUPRC and F1 on the test waterfalls, and the **batch-1 latency on the T4, including the copies** to and from the GPU.

This is a **starting point, not a good design**: a small, simple model that runs end to end (data → training → the chit's metric → its cost on the target),
so your team has something to beat. Every number it prints is yours to improve. Run time: ~10 min on a laptop CPU (measured), plus the first download; training uses the GPU automatically if the runtime has one.

> **Before you start:** **File → Save a copy in Drive**, then run the cells in order.

> **Runtime → Change runtime type → T4 GPU** before running: the latency budget is defined on Colab's T4. (Without a GPU everything still runs, and the latency printed is the CPU's, labelled as such.)

In [ ]:
# --- Colab setup: run once per session (~1 min) ---
import os, sys, glob, subprocess
IN_COLAB = "google.colab" in sys.modules
USE_DRIVE = False       # True: keep the chit's data on your Google Drive between sessions
LAB = "capstone/baselines"
if IN_COLAB:
    ROOT = "/content/aims-fastml-labs"
    if not os.path.exists(ROOT):
        bundles = glob.glob("/content/*bundle*.zip")
        if bundles:
            subprocess.run(["unzip", "-q", "-o", bundles[0], "-d", ROOT], check=True)
        else:
            subprocess.run(["git", "clone", "-q", "https://github.com/SagarA17/aims-fastml-labs.git", ROOT], check=True)
    else:
        subprocess.run(["git", "-C", ROOT, "pull", "-q"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *["onnx"]], check=True)
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        os.environ["FASTML_DATA"] = "/content/drive/MyDrive/fastml_data"
    os.chdir(f"{ROOT}/{LAB}")
print("working directory:", os.getcwd())

In [ ]:
import os, sys, time, json, warnings
sys.path.append("../.."); warnings.filterwarnings("ignore")
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from common.budget import profile, fits
from capstone.baselines.tools import FAST, fit, predict, to_onnx, ort_session, ort_int8, ort_predict, ort_latency, mcu_latency, energy_uj, report
torch.manual_seed(0); np.random.seed(0)
os.makedirs("my_models", exist_ok=True)
T0 = time.time()

In [ ]:
from common.timing import measure, cpu_call, gpu_call
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEV, torch.cuda.get_device_name(0) if DEV == "cuda" else "")

## 1. Data
One waterfall = 512 × 512 amplitudes; the mask marks RFI pixels (2.75% of them).

In [ ]:
from capstone.data.hera import load, threshold_flagger, auprc, best_f1, f1_at
X_tr, y_tr = load("train", max_items=40 if FAST else None)
X_va, y_va = load("val", max_items=10 if FAST else None)
X_te, y_te = load("test", max_items=20 if FAST else None)
print("train", X_tr.shape, "| val", X_va.shape, "| test", X_te.shape, "| RFI fraction (train):", round(float(y_tr.mean()), 4))

## 2. The classical baseline: a robust threshold
For every row, the median and the MAD of log|V|; a pixel's score is how many robust standard deviations it sits above the median.
The threshold is chosen on **val** (best F1), then applied to **test**.

In [ ]:
FLAG_THR = best_f1(y_va, threshold_flagger(X_va))[1]
s_te = threshold_flagger(X_te)
FLAG_AP, FLAG_F1 = auprc(y_te, s_te), f1_at(y_te, s_te, FLAG_THR)
print(f"threshold flagger, test: AUPRC {FLAG_AP:.4f}, F1 {FLAG_F1:.4f} (threshold {FLAG_THR:.2f}, chosen on val)")

## 3. A tiny U-Net, costed before training
The log and the normalisation are **part of the model** (`Prep`), so the timed call below includes them. One downsampling level, 8 channels.

In [ ]:
LX = np.log10(X_tr[:, :, ::8, ::8] + 1e-6)                    # normalisation constants from the training split
MU, SD = float(LX.mean()), float(LX.std())

class Prep(nn.Module):
    def __init__(self, mu, sd):
        super().__init__(); self.mu, self.sd = mu, sd
    def forward(self, x):
        return (torch.log10(x + 1e-6) - self.mu) / self.sd

def block(a, b, stride=1):
    return nn.Sequential(nn.Conv2d(a, b, 3, stride=stride, padding=1), nn.ReLU(), nn.Conv2d(b, b, 3, padding=1), nn.ReLU())

class TinyUNet(nn.Module):
    def __init__(self, c=8):
        super().__init__()
        self.prep = Prep(MU, SD)
        self.enc, self.down = block(1, c), block(c, 2 * c, stride=2)
        self.up = nn.Sequential(nn.Upsample(scale_factor=2), nn.Conv2d(2 * c, c, 3, padding=1), nn.ReLU())
        self.dec, self.out = block(2 * c, c), nn.Conv2d(c, 1, 1)
    def forward(self, x):
        e = self.enc(self.prep(x))
        return self.out(self.dec(torch.cat([e, self.up(self.down(e))], 1)))   # logits per pixel

model = TinyUNet()
p = profile(model, (1, 512, 512), bytes_per_weight=4, bytes_per_act=4)   # fp32 on the GPU
print(f"params {p['params']:,} | MACs per waterfall {p['macs'] / 1e9:.2f} G | peak activation (fp32) {p['peak_act_bytes'] / 1e6:.1f} MB")

## 4. Train on 128 × 128 tiles (the network is fully convolutional), evaluate on full waterfalls

In [ ]:
tile = lambda A: A.reshape(len(A), 1, 4, 128, 4, 128).transpose(0, 2, 4, 1, 3, 5).reshape(-1, 1, 128, 128)
T_X, T_y = tile(X_tr), tile(y_tr).astype(np.float32)
POS_W = torch.tensor(5.0, device=DEV)
fit(model, T_X, T_y, epochs=1 if FAST else 8, bs=32, lr=2e-3, device=DEV,
    loss_fn=lambda out, yb: F.binary_cross_entropy_with_logits(out, yb.float(), pos_weight=POS_W))
p_va, p_te = predict(model, X_va, bs=4, device=DEV), predict(model, X_te, bs=4, device=DEV)
THR = best_f1(y_va, p_va)[1]
NET_AP, NET_F1 = auprc(y_te, p_te), f1_at(y_te, p_te, THR)
print(f"tiny U-Net, test: AUPRC {NET_AP:.4f}, F1 {NET_F1:.4f}   (flagger: AUPRC {FLAG_AP:.4f}, F1 {FLAG_F1:.4f})")

## 5. Latency at batch 1, including the copies
`gpu_call` copies one waterfall (1 MB) to the GPU, runs the model and copies the scores back; `measure` repeats it and reports the median and the p99.

In [ ]:
x1 = torch.from_numpy(X_te[:1])
model = model.to(DEV).eval()
if DEV == "cuda":
    torch.cuda.reset_peak_memory_stats()
    LAT = measure(gpu_call(model, x1), cuda=True)
    FLAG_LAT = measure(lambda: threshold_flagger(x1.to("cuda")).cpu(), cuda=True)
    MEM_GB = torch.cuda.max_memory_allocated() / 1e9
    where = "T4 GPU, incl. copies"
else:
    torch.set_num_threads(1)
    LAT = measure(cpu_call(model, x1), n=50, warmup=5, repeats=3)
    FLAG_LAT = measure(lambda: threshold_flagger(x1), n=50, warmup=5, repeats=3)
    MEM_GB = float("nan")
    where = "CPU (1 thread): NOT the target; run on a T4 for the real number"
print(f"tiny U-Net: median {LAT['median_us'] / 1e3:.2f} ms, p99 {LAT['p99_us'] / 1e3:.2f} ms ({where}); peak GPU memory {MEM_GB:.3f} GB")
print(f"flagger:    median {FLAG_LAT['median_us'] / 1e3:.2f} ms, p99 {FLAG_LAT['p99_us'] / 1e3:.2f} ms")

In [ ]:
ROW = report("3 hera", (len(X_tr), len(X_va), len(X_te)), (1, 512, 512),
             f"U-Net AUPRC {NET_AP:.4f} F1 {NET_F1:.4f} | flagger AUPRC {FLAG_AP:.4f} F1 {FLAG_F1:.4f}", p,
             f"{where}: U-Net median {LAT['median_us'] / 1e3:.2f} / p99 {LAT['p99_us'] / 1e3:.2f} ms, flagger p99 {FLAG_LAT['p99_us'] / 1e3:.2f} ms; GPU mem {MEM_GB:.3f} GB",
             "CC BY 4.0 (Zenodo, confirmed)", extra=dict(minutes=round((time.time() - T0) / 60, 1)))

---
**Where next.** Read your chit card and `targets.md` again with these numbers in front of you: which budget binds first? The levers are listed in `targets.md`.